In [ ]:
!pip uninstall pinecone-client google-generativeai -y
!pip install --upgrade pip
!pip install pinecone-client==3.2.2
!pip install google-generativeai==0.8.3
!pip install tqdm

In [ ]:
import requests
import json
import re
import time
from typing import List, Dict

from pinecone import Pinecone, ServerlessSpec
from tqdm.auto import tqdm


# ================= CONFIG =================

OPENROUTER_API_KEY = ""
PINECONE_API_KEY = ""

INDEX_NAME = "respiratory-medical-rag"

MODEL = "nvidia/llama-nemotron-embed-vl-1b-v2:free"

DIMENSION = 2048
METRIC = "cosine"

KNOWLEDGE_BASE_FILE = "/content/respiratory_knowledge_base.txt"

FORCE_RECREATE_INDEX = True


# ================= CONNECT PINECONE =================

print("\n🔧 Connecting Pinecone...")

pc = Pinecone(api_key=PINECONE_API_KEY)

existing = [i.name for i in pc.list_indexes()]
print("Existing indexes:", existing)

if INDEX_NAME in existing:

    info = pc.describe_index(INDEX_NAME)
    current_dimension = info.dimension

    if current_dimension != DIMENSION:

        print("⚠️ Dimension mismatch")

        if FORCE_RECREATE_INDEX:

            print("Deleting old index...")
            pc.delete_index(INDEX_NAME)
            time.sleep(5)

            print("Creating new index...")

            pc.create_index(
                name=INDEX_NAME,
                dimension=DIMENSION,
                metric=METRIC,
                spec=ServerlessSpec(
                    cloud="aws",
                    region="us-east-1"
                )
            )

            time.sleep(10)

else:

    print("Creating new index...")

    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

    time.sleep(10)


index = pc.Index(INDEX_NAME)

print("Index ready:", index.describe_index_stats())


# ================= EMBEDDING =================

def create_embedding(text: str, retries: int = 3):

    url = "https://openrouter.ai/api/v1/embeddings"

    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": MODEL,
        "input": text
    }

    for attempt in range(retries):

        try:

            r = requests.post(url, headers=headers, json=payload)

            if r.status_code != 200:
                raise Exception(r.text)

            data = r.json()

            return data["data"][0]["embedding"]

        except Exception as e:

            if attempt < retries - 1:
                print("Retry embedding:", str(e)[:100])
                time.sleep(2)
            else:
                print("Embedding failed:", e)
                return None


# ================= TEXT CLEAN =================

def clean_text(text: str):

    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)

    return text.strip()


# ================= CATEGORY =================

def detect_category(text: str):

    text = text.lower()

    patterns = {
        "viem_phoi": r"\bviêm phổi\b|\bpneumonia\b",
        "hen_suyen": r"\bhen\b|\basthma\b",
        "copd": r"\bcopd\b",
        "kho_tho": r"\bkhó thở\b|\bdyspnea\b",
        "ho": r"\bho\b|\bcough\b",
        "covid": r"\bcovid\b",
        "cam_cum": r"\bcúm\b|\binfluenza\b",
        "dau_nguc": r"\bđau ngực\b",
        "thuoc": r"\bthuốc\b|\bparacetamol\b"
    }

    for k, p in patterns.items():
        if re.search(p, text):
            return k

    return "general"


# ================= SEVERITY =================

def detect_severity(text: str):

    text = text.lower()

    if re.search(r'115|cấp cứu|tím tái|nguy hiểm', text):
        return "critical"

    if re.search(r'nặng|dữ dội|sốt cao', text):
        return "high"

    if re.search(r'nhẹ|thông thường', text):
        return "low"

    return "medium"


# ================= KEYWORDS =================

def extract_keywords(text: str):

    terms = [
        "sốt",
        "ho",
        "khó thở",
        "đau ngực",
        "đờm",
        "tím tái",
        "mệt mỏi",
        "sổ mũi",
        "đau họng",
        "viêm phổi",
        "hen",
        "copd",
        "covid",
        "cúm",
        "paracetamol"
    ]

    text_lower = text.lower()

    return [t for t in terms if t in text_lower]


# ================= PARSE KNOWLEDGE BASE =================

def parse_knowledge_base(path: str):

    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()

    sections = re.split(r'\n(?=## )', raw)

    chunks = []

    for sec_id, sec in enumerate(sections):

        if not sec.startswith("##"):
            continue

        lines = sec.split("\n")

        section_title = lines[0].replace("#", "").strip()

        subsections = re.split(r'\n(?=### )', "\n".join(lines[1:]))

        for sub_id, sub in enumerate(subsections):

            sub = sub.strip()

            if not sub:
                continue

            sub_lines = sub.split("\n")

            if sub_lines[0].startswith("###"):
                subsection_title = sub_lines[0].replace("#", "").strip()
                body = "\n".join(sub_lines[1:])
            else:
                subsection_title = "Tổng quan"
                body = "\n".join(sub_lines)

            body = clean_text(body)

            if len(body) < 40:
                continue

            # ⭐ Semantic chunk format
            chunk_text = f"""
Bệnh hô hấp: {section_title}
Chủ đề: {subsection_title}

Nội dung:
{body}
"""

            category = detect_category(chunk_text)
            severity = detect_severity(chunk_text)
            keywords = extract_keywords(chunk_text)

            chunks.append({
                "id": f"kb_{sec_id}_{sub_id}",
                "text": chunk_text,
                "metadata": {
                    "section": section_title,
                    "subsection": subsection_title,
                    "category": category,
                    "severity": severity,
                    "keywords": keywords,
                    "language": "vi",
                    "text": chunk_text[:2000]
                }
            })

    return chunks


# ================= IMPORT =================

def import_to_pinecone(chunks: List[Dict], batch_size=10):

    vectors = []

    print("Importing", len(chunks), "chunks")

    for chunk in tqdm(chunks):

        emb = create_embedding(chunk["text"])

        if emb is None:
            continue

        vectors.append({
            "id": chunk["id"],
            "values": emb,
            "metadata": chunk["metadata"]
        })

        if len(vectors) >= batch_size:

            index.upsert(vectors=vectors)
            vectors = []

            time.sleep(0.5)

    if vectors:
        index.upsert(vectors=vectors)

    print("Import done")
    print(index.describe_index_stats())


# ================= TEST QUERY =================

def test_query(q: str):

    print("\n")
    print("=" * 60)
    print("Query:", q)
    print("=" * 60)

    emb = create_embedding(q)

    res = index.query(
        vector=emb,
        top_k=5,
        include_metadata=True
    )

    for i, m in enumerate(res["matches"], 1):

        md = m["metadata"]

        print("\nResult", i)
        print("Score:", m["score"])
        print("Section:", md["section"])
        print("Subsection:", md["subsection"])
        print("Category:", md["category"])
        print("Severity:", md["severity"])
        print("Text:", md["text"][:200], "...")


# ================= MAIN =================

if __name__ == "__main__":

    print("\n==============================")
    print("RESPIRATORY KB IMPORTER")
    print("==============================")

    chunks = parse_knowledge_base(KNOWLEDGE_BASE_FILE)

    print("Chunks:", len(chunks))

    for c in chunks[:3]:
        print(c["id"], c["metadata"]["category"], c["metadata"]["severity"])

    import_to_pinecone(chunks)

    print("\nTesting queries...")

    tests = [
        "Tôi bị ho khan",
        "Khó thở nghiêm trọng",
        "Thuốc paracetamol",
        "Sốt cao trên 38.5°C",
        "Triệu chứng viêm phổi"
    ]

    for t in tests:
        test_query(t)